# Assignment 1 — Build a Custom Missing-Value Imputer

**Course:** Feature Engineering & MLOps  
**Assignment No.:** 1  
**Topic:** Custom Imputer Class (Missing Value Handling)

## Objective

The goal of this assignment is to build a custom Python imputer that follows the same `fit()` / `transform()` pattern used by scikit-learn transformers.

The imputer will:

- automatically identify numeric and non-numeric columns,
- learn fill values using **training data only**,
- support median/mean imputation for numeric columns,
- use the most frequent value for non-numeric columns,
- optionally add missing-value indicator columns,
- protect against calling `transform()` before `fit()`, and
- support an optional per-column strategy override as the bonus task.

The supplied `student_performance_raw.csv` dataset is used throughout the notebook.

## Assignment Covered

This notebook follows the assignment brief from the provided PDF.

### Required work
1. Design `CustomImputer` using `BaseEstimator` and `TransformerMixin`.
2. Implement `fit()` and `transform()`.
3. Automatically detect numeric vs. non-numeric columns.
4. Fit only on the 80% training split.
5. Transform both train and test data using the fitted statistics.
6. Verify that no missing values remain in the imputed dataset.
7. Add missingness indicators for columns that were missing during training.
8. Compare numeric statistics before and after imputation.
9. Compare the custom numeric fill values with scikit-learn's `SimpleImputer`.
10. Answer all three reflection questions.
11. Demonstrate the optional **per-column strategy override** bonus.

In [29]:
# Imports and basic setup
import os
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted
RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("Libraries imported successfully.")
print("Random state:", RANDOM_STATE)

Libraries imported successfully.
Random state: 42


## 1. Load the Dataset

The assignment specifies `student_performance_raw.csv`. The path resolver below first checks the standard course-repository location and then the local notebook directory, so the same notebook can be used both in the submitted repository and with the supplied dataset file.

In [30]:
# Locate the supplied dataset
candidate_paths = [
    "/content/student_performance_raw.csv"
]

DATA_PATH = next((path for path in candidate_paths if os.path.exists(path)), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "student_performance_raw.csv was not found. "
        "Place it in data/raw/ or beside this notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Loaded dataset from:", DATA_PATH)
print("Dataset shape:", df.shape)

display(df.head())

Loaded dataset from: /content/student_performance_raw.csv
Dataset shape: (600, 17)


,student_id,city,city_tier,course,batch_type,age,enrollment_date,attendance_pct,weekly_study_hours,income_bracket,prev_exam_score,mock_test_1,mock_test_2,mock_test_3,doubt_sessions_attended,feedback_text,final_score
0,1447,Patna,2,JEE Main,Weekday,18,2026-05-27,91.1000,0.8000,<5L,65.2000,68.1000,59.0000,72.2000,1,Need more practice sheets for weak topics,42.0000
1,1405,Patna,2,NEET,Weekday,16,2024-07-08,91.9000,2.4000,NaN,64.9000,82.0000,88.4000,100.0000,3,Would like more one-on-one mentoring,62.0000
2,1510,Mumbai,1,JEE Main,Weekday,18,2025-03-17,67.1000,6.7000,5-10L,90.1000,86.5000,91.1000,76.4000,5,"Great teaching pace, doubts cleared quickly",56.4000
3,1456,Delhi,1,NEET,Weekday,18,2025-06-16,70.7000,2.0000,10-20L,29.7000,19.4000,24.9000,NaN,3,Need more practice sheets for weak topics,31.7000
4,1202,Patna,2,JEE Advanced,Weekday,17,2026-04-16,69.3000,6.5000,>20L,70.0000,63.8000,74.5000,69.1000,4,Need more practice sheets for weak topics,44.5000


In [31]:
# Dataset structure and missing values
print("Data types:")
display(df.dtypes.to_frame("dtype"))
print("Missing values:")
missing_summary = (
    df.isna().sum()
      .rename("missing_count")
      .to_frame()
      .assign(missing_pct=lambda x: 100 * x["missing_count"] / len(df))
      .query("missing_count > 0")
)

display(missing_summary)

Data types:


,dtype
student_id,int64
city,object
city_tier,int64
course,object
batch_type,object
age,int64
enrollment_date,object
attendance_pct,float64
weekly_study_hours,float64
income_bracket,object


Missing values:


,missing_count,missing_pct
weekly_study_hours,36,6.0000
income_bracket,30,5.0000
prev_exam_score,25,4.1667
mock_test_3,21,3.5000
feedback_text,68,11.3333


### Missingness Profile from the Assignment Brief

The assignment identifies the missing-value mechanisms as follows:

| Column | Missingness type | Data type |
|---|---|---|
| `weekly_study_hours` | MCAR (random) | Numeric |
| `prev_exam_score` | MAR (linked to `attendance_pct`) | Numeric |
| `mock_test_3` | MNAR (linked to its own low values) | Numeric |
| `income_bracket` | Missing categorical values | Categorical |
| `feedback_text` | Missing free-text values | Text |

The custom class does not need these mechanisms to calculate the fill values. They are important for understanding **why missingness itself can contain useful information**, which is why the indicator feature is included.

## 2. Design the `CustomImputer` Class

### Design choices

- **`BaseEstimator` + `TransformerMixin`** make the class follow the scikit-learn estimator API.
- **Median** is the default numeric strategy because it is less sensitive to extreme values than the mean.
- **Most frequent** is used for non-numeric columns, including the free-text column in this dataset.
- Statistics are learned only inside `fit()`.
- `transform()` uses the stored fitted values and never recalculates them.
- Missingness indicators are created **before** filling values, so they capture the original missingness pattern.
- The optional `column_overrides` dictionary allows a specific column to use a different strategy.

In [32]:
class CustomImputer(BaseEstimator, TransformerMixin):
    """
    Custom missing-value imputer following the scikit-learn transformer API.

    The class automatically detects numeric and non-numeric columns. During
    fit(), it learns one fill value per column from the supplied training data.
    Numeric columns use either mean or median, while non-numeric columns use
    the most frequent value by default.

    Parameters
    ----------
    numeric_strategy : str, default="median"
        Strategy for numeric columns. Must be "mean" or "median".

    categorical_strategy : str, default="most_frequent"
        Strategy for non-numeric columns. The assignment requires
        "most_frequent".

    add_missing_indicator : bool, default=True
        Whether to add <column>_was_missing indicator columns for columns
        that contained missing values during fit().

    column_overrides : dict or None, default=None
        Optional mapping from column name to an individual strategy.
        Supported strategies are "mean", "median", and "most_frequent".

    Returns
    -------
    CustomImputer
        A configured imputer object.
    """

    def __init__(
        self,
        numeric_strategy="median",
        categorical_strategy="most_frequent",
        add_missing_indicator=True,
        column_overrides=None,
    ):
        if numeric_strategy not in {"mean", "median"}:
            raise ValueError("numeric_strategy must be 'mean' or 'median'.")

        if categorical_strategy != "most_frequent":
            raise ValueError(
                "categorical_strategy must be 'most_frequent' for this assignment."
            )

        if not isinstance(add_missing_indicator, bool):
            raise TypeError("add_missing_indicator must be a boolean.")

        if column_overrides is not None and not isinstance(column_overrides, dict):
            raise TypeError("column_overrides must be a dictionary or None.")

        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator
        self.column_overrides = column_overrides

    def _get_fill_value(self, series, strategy):
        """Calculate the fill value for one pandas Series."""
        if strategy == "mean":
            return series.mean()

        if strategy == "median":
            return series.median()

        if strategy == "most_frequent":
            mode = series.mode(dropna=True)
            if mode.empty:
                return np.nan
            return mode.iloc[0]

        raise ValueError(f"Unsupported strategy: {strategy}")

    def fit(self, X, y=None):
        """
        Learn and store fill values from the training DataFrame.

        Parameters
        ----------
        X : pandas.DataFrame
            Training data from which imputation statistics are learned.

        y : optional
            Ignored. Included for scikit-learn API compatibility.

        Returns
        -------
        self
            The fitted imputer.
        """
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        self.feature_names_in_ = X.columns.tolist()
        self.column_types_ = {}
        self.strategies_ = {}
        self.fill_values_ = {}
        self.missing_indicator_columns_ = []

        # Check that override keys actually exist in the training data.
        if self.column_overrides is not None:
            unknown_overrides = [
                column for column in self.column_overrides
                if column not in X.columns
            ]
            if unknown_overrides:
                raise ValueError(
                    "column_overrides contains unknown columns: "
                    + ", ".join(unknown_overrides)
                )

        for column in X.columns:
            series = X[column]

            if pd.api.types.is_numeric_dtype(series):
                self.column_types_[column] = "numeric"
                default_strategy = self.numeric_strategy
            else:
                self.column_types_[column] = "categorical/text"
                default_strategy = self.categorical_strategy

            strategy = (
                self.column_overrides.get(column, default_strategy)
                if self.column_overrides is not None
                else default_strategy
            )

            if strategy not in {"mean", "median", "most_frequent"}:
                raise ValueError(
                    f"Unsupported strategy '{strategy}' for column '{column}'."
                )

            if strategy in {"mean", "median"} and not pd.api.types.is_numeric_dtype(series):
                raise ValueError(
                    f"Strategy '{strategy}' requires a numeric column; "
                    f"'{column}' is non-numeric."
                )

            self.strategies_[column] = strategy
            self.fill_values_[column] = self._get_fill_value(series, strategy)

            if series.isna().any():
                self.missing_indicator_columns_.append(column)

        return self

    def transform(self, X):
        """
        Fill missing values using statistics learned during fit().

        Parameters
        ----------
        X : pandas.DataFrame
            Data to transform. It must have the same columns seen during fit().

        Returns
        -------
        pandas.DataFrame
            A copy of X with missing values filled and, when enabled, missing
            indicator columns added.
        """
        check_is_fitted(
            self,
            attributes=[
                "fill_values_",
                "feature_names_in_",
                "column_types_",
                "strategies_",
            ],
        )

        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        missing_columns = [
            column for column in self.feature_names_in_
            if column not in X.columns
        ]
        if missing_columns:
            raise ValueError(
                "Transform data is missing columns seen during fit: "
                + ", ".join(missing_columns)
            )

        extra_columns = [
            column for column in X.columns
            if column not in self.feature_names_in_
        ]
        if extra_columns:
            raise ValueError(
                "Transform data contains columns not seen during fit: "
                + ", ".join(extra_columns)
                + ". Refit the imputer or align the schema before transforming."
            )

        X_out = X.copy()
        if self.add_missing_indicator:
            for column in self.missing_indicator_columns_:
                X_out[f"{column}_was_missing"] = X[column].isna().astype(int)

        for column, fill_value in self.fill_values_.items():
            if pd.isna(fill_value):
                continue

            X_out[column] = X_out[column].fillna(fill_value)

        return X_out


### 2.1 Guard-Rail Check

A transformer should not be usable before it has learned its statistics. `check_is_fitted()` provides the required scikit-learn-style guard rail.

In [33]:
# transform() before fit() should raise NotFittedError
unfitted_imputer = CustomImputer()
try:
    unfitted_imputer.transform(df)
except Exception as exc:
    print(f"Guard-rail passed: {type(exc).__name__}")
    print("Message:", exc)

Guard-rail passed: NotFittedError
Message: This CustomImputer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


## 3. Train/Test Split — Strict Train-Only Fitting

The assignment requires an **80/20 split**. The imputer is fitted only on the training data. This is important because the learned mean, median, and mode are preprocessing statistics; allowing test data to influence them would be a form of data leakage.

In [34]:
# 80/20 train-test split
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Full dataset :", df.shape)
print("Training set :", train_df.shape)
print("Test set     :", test_df.shape)

print("\nMissing values in each split:")
split_missing = pd.DataFrame({
    "train_missing": train_df.isna().sum(),
    "test_missing": test_df.isna().sum(),
})
display(split_missing.query("train_missing > 0 or test_missing > 0"))

Full dataset : (600, 17)
Training set : (480, 17)
Test set     : (120, 17)

Missing values in each split:


,train_missing,test_missing
weekly_study_hours,25,11
income_bracket,24,6
prev_exam_score,23,2
mock_test_3,15,6
feedback_text,54,14


## 4. Fit the Custom Imputer on Training Data Only

The default configuration follows the assignment:

- numeric columns → **median**
- non-numeric columns → **most frequent**
- missing indicators → **enabled**

Notice that the learned values below come from `train_df` only.

In [35]:
# Fit ONLY on the training split
imputer = CustomImputer(
    numeric_strategy="median",
    categorical_strategy="most_frequent",
    add_missing_indicator=True,
)

imputer.fit(train_df)

print("Automatically detected column types:")
display(pd.Series(imputer.column_types_, name="detected_type").to_frame())

print("Strategies used:")
display(pd.Series(imputer.strategies_, name="strategy").to_frame())

print("Learned fill values (training data only):")
display(pd.Series(imputer.fill_values_, name="fill_value").to_frame())

print("Columns that had missing values during fit():")
print(imputer.missing_indicator_columns_)

Automatically detected column types:


,detected_type
student_id,numeric
city,categorical/text
city_tier,numeric
course,categorical/text
batch_type,categorical/text
age,numeric
enrollment_date,categorical/text
attendance_pct,numeric
weekly_study_hours,numeric
income_bracket,categorical/text


Strategies used:


,strategy
student_id,median
city,most_frequent
city_tier,median
course,most_frequent
batch_type,most_frequent
age,median
enrollment_date,most_frequent
attendance_pct,median
weekly_study_hours,median
income_bracket,most_frequent


Learned fill values (training data only):


,fill_value
student_id,1282.0000
city,Mumbai
city_tier,1.0000
course,NEET
batch_type,Weekday
age,17.0000
enrollment_date,2024-10-23
attendance_pct,78.3500
weekly_study_hours,5.0000
income_bracket,5-10L


Columns that had missing values during fit():
['weekly_study_hours', 'income_bracket', 'prev_exam_score', 'mock_test_3', 'feedback_text']


## 5. Transform Training and Test Data

The same fitted imputer is now applied to both datasets. No statistic is recalculated from the test set.

In [36]:
# Transform both splits using the same fitted imputer
train_imputed = imputer.transform(train_df)
test_imputed = imputer.transform(test_df)

indicator_cols = [
    f"{column}_was_missing"
    for column in imputer.missing_indicator_columns_
]

print("Original train shape :", train_df.shape)
print("Imputed train shape  :", train_imputed.shape)
print("Original test shape  :", test_df.shape)
print("Imputed test shape   :", test_imputed.shape)

print("\nIndicator columns created:")
print(indicator_cols)

print("\nSample of the transformed training data:")
display(train_imputed.head())

Original train shape : (480, 17)
Imputed train shape  : (480, 22)
Original test shape  : (120, 17)
Imputed test shape   : (120, 22)

Indicator columns created:
['weekly_study_hours_was_missing', 'income_bracket_was_missing', 'prev_exam_score_was_missing', 'mock_test_3_was_missing', 'feedback_text_was_missing']

Sample of the transformed training data:


,student_id,city,city_tier,course,batch_type,age,enrollment_date,attendance_pct,weekly_study_hours,income_bracket,prev_exam_score,mock_test_1,mock_test_2,mock_test_3,doubt_sessions_attended,feedback_text,final_score,weekly_study_hours_was_missing,income_bracket_was_missing,prev_exam_score_was_missing,mock_test_3_was_missing,feedback_text_was_missing
145,1551,Pune,1,JEE Main,Weekend,17,2024-12-17,59.8000,2.0000,5-10L,56.5000,58.3000,46.2000,54.5000,5,Would like more one-on-one mentoring,39.3000,0,0,0,0,0
9,1166,Kota,2,JEE Main,Weekday,15,2025-08-19,72.9000,5.0000,10-20L,57.8000,46.6000,40.9000,34.9000,6,Need more practice sheets for weak topics,43.2000,1,0,0,0,0
375,1367,Hyderabad,2,Foundation,Weekday,17,2026-02-14,72.1000,6.2000,5-10L,45.1000,52.0000,59.1000,65.9000,1,Need more practice sheets for weak topics,44.0000,0,0,0,0,0
523,1562,Delhi,1,NEET,Weekday,19,2025-10-04,82.5000,1.1000,5-10L,54.3000,58.7000,63.8000,74.0000,4,Mock tests really helped identify gaps,39.6000,0,0,0,0,0
188,1575,Lucknow,2,NEET,Weekend,15,2025-05-29,98.0000,2.8000,<5L,78.8000,73.8000,78.6000,87.9000,3,"Great teaching pace, doubts cleared quickly",55.0000,0,0,0,0,0


## 6. Verification — No Missing Values Remain

The assignment specifically asks for a programmatic verification. The check below examines **all original columns** after transformation and uses assertions so the notebook fails loudly if imputation has not worked.

In [37]:
# Programmatic verification
original_columns = imputer.feature_names_in_

train_missing_after = train_imputed[original_columns].isna().sum()
test_missing_after = test_imputed[original_columns].isna().sum()

print("Remaining missing values in original columns — TRAIN:")
display(train_missing_after[train_missing_after > 0].to_frame("missing_count"))

print("Remaining missing values in original columns — TEST:")
display(test_missing_after[test_missing_after > 0].to_frame("missing_count"))

assert train_imputed[original_columns].isna().sum().sum() == 0
assert test_imputed[original_columns].isna().sum().sum() == 0

print("PASS: No missing values remain in any original column after imputation.")

Remaining missing values in original columns — TRAIN:


,missing_count


Remaining missing values in original columns — TEST:


,missing_count


PASS: No missing values remain in any original column after imputation.


## 7. Missingness Indicators

For every column that had missing values in the **training split**, the transformer creates a binary `<column>_was_missing` feature.

- `1` → the original value was missing
- `0` → the original value was present

This is useful because ordinary imputation replaces the missing value and can otherwise erase the fact that it was missing in the first place.

In [38]:
# Verify that the indicator columns preserve the original missingness pattern
indicator_summary = pd.DataFrame({
    "indicator_column": indicator_cols,
    "train_missing_count": [
        train_df[column].isna().sum()
        for column in imputer.missing_indicator_columns_
    ],
    "indicator_sum_after_transform": [
        train_imputed[column].sum()
        for column in indicator_cols
    ],
})

display(indicator_summary)

for source_col, indicator_col in zip(
    imputer.missing_indicator_columns_,
    indicator_cols
):
    assert train_imputed[indicator_col].sum() == train_df[source_col].isna().sum()
    assert set(train_imputed[indicator_col].unique()).issubset({0, 1})

print("PASS: Missing indicators correctly preserve the original missingness pattern.")

,indicator_column,train_missing_count,indicator_sum_after_transform
0,weekly_study_hours_was_missing,25,25
1,income_bracket_was_missing,24,24
2,prev_exam_score_was_missing,23,23
3,mock_test_3_was_missing,15,15
4,feedback_text_was_missing,54,54


PASS: Missing indicators correctly preserve the original missingness pattern.


## 8. Numeric Descriptive Statistics — Before vs. After Imputation

The assignment asks for the **mean and standard deviation of each numeric column before and after imputation**.

These statistics are calculated on the training split. Columns without missing values should remain unchanged, while columns with missing values can show a change because their missing observations have been replaced by the training-set median.

In [39]:
# Compare every numeric column in the training split
numeric_cols = train_df.select_dtypes(include=np.number).columns.tolist()

stats_rows = []

for column in numeric_cols:
    before_mean = train_df[column].mean()
    before_std = train_df[column].std()

    after_mean = train_imputed[column].mean()
    after_std = train_imputed[column].std()

    stats_rows.append({
        "column": column,
        "missing_in_train": train_df[column].isna().sum(),
        "mean_before": before_mean,
        "mean_after": after_mean,
        "std_before": before_std,
        "std_after": after_std,
        "mean_change": after_mean - before_mean,
        "std_change": after_std - before_std,
    })

stats_comparison = pd.DataFrame(stats_rows)

display(stats_comparison.round(4))

,column,missing_in_train,mean_before,mean_after,std_before,std_after,mean_change,std_change
0,student_id,0,1290.5625,1290.5625,173.8254,173.8254,0.0000,0.0000
1,city_tier,0,1.4021,1.4021,0.4908,0.4908,0.0000,0.0000
2,age,0,16.9562,16.9562,1.7315,1.7315,0.0000,0.0000
3,attendance_pct,0,78.5462,78.5462,13.4596,13.4596,0.0000,0.0000
4,weekly_study_hours,25,6.0642,6.0088,4.3503,4.2418,-0.0554,-0.1084
5,prev_exam_score,23,65.8245,65.8377,14.3708,14.0217,0.0132,-0.3491
6,mock_test_1,0,65.9452,65.9452,16.5662,16.5662,0.0000,0.0000
7,mock_test_2,0,67.8862,67.8862,18.1088,18.1088,0.0000,0.0000
8,mock_test_3,15,70.7015,70.7202,19.0773,18.7765,0.0187,-0.3008
9,doubt_sessions_attended,0,4.0042,4.0042,2.0259,2.0259,0.0000,0.0000


### What changed?

For numeric columns with missing values, the missing observations are replaced with the training-set median. The mean can move slightly because new values have been inserted, while the standard deviation generally decreases because several observations are replaced by the same central value.

Numeric columns that had no missing values are unchanged by the imputer. This is a useful sanity check that the transformer is only modifying values where imputation is actually needed.

## 9. Sanity Check Against scikit-learn `SimpleImputer`

To verify the core logic, the assignment asks for an equivalent `SimpleImputer(strategy="median")` on the **same numeric training columns**.

The custom class and scikit-learn should learn the same median values.

In [40]:
# Equivalent SimpleImputer, also fitted only on training data
sk_imputer = SimpleImputer(strategy="median")
sk_imputer.fit(train_df[numeric_cols])

custom_numeric_values = np.array(
    [imputer.fill_values_[column] for column in numeric_cols],
    dtype=float
)

sk_numeric_values = sk_imputer.statistics_.astype(float)

sanity_check = pd.DataFrame({
    "column": numeric_cols,
    "custom_fill_value": custom_numeric_values,
    "simple_imputer_value": sk_numeric_values,
})

sanity_check["absolute_difference"] = np.abs(
    sanity_check["custom_fill_value"]
    - sanity_check["simple_imputer_value"]
)

display(sanity_check.round(6))

assert np.allclose(
    custom_numeric_values,
    sk_numeric_values,
    equal_nan=True
)

print("PASS: CustomImputer and SimpleImputer produce identical numeric median fill values.")

,column,custom_fill_value,simple_imputer_value,absolute_difference
0,student_id,1282.0000,1282.0000,0.0000
1,city_tier,1.0000,1.0000,0.0000
2,age,17.0000,17.0000,0.0000
3,attendance_pct,78.3500,78.3500,0.0000
4,weekly_study_hours,5.0000,5.0000,0.0000
5,prev_exam_score,66.1000,66.1000,0.0000
6,mock_test_1,66.5500,66.5500,0.0000
7,mock_test_2,66.3500,66.3500,0.0000
8,mock_test_3,71.3000,71.3000,0.0000
9,doubt_sessions_attended,4.0000,4.0000,0.0000


PASS: CustomImputer and SimpleImputer produce identical numeric median fill values.


## 10. Categorical/Text Imputation Check

The automatic type detector treats non-numeric columns as `categorical/text`. The assignment requires the most frequent value for categorical data, so the same rule is applied to columns such as `income_bracket` and `feedback_text`.

The following check compares the custom fill values against the training-set mode.

In [41]:
# Verify non-numeric fill values against the training-set mode
non_numeric_cols = train_df.select_dtypes(exclude=np.number).columns.tolist()

categorical_check_rows = []

for column in non_numeric_cols:
    mode = train_df[column].mode(dropna=True).iloc[0]

    categorical_check_rows.append({
        "column": column,
        "custom_fill_value": imputer.fill_values_[column],
        "training_mode": mode,
        "matches": imputer.fill_values_[column] == mode,
    })

categorical_check_df = pd.DataFrame(categorical_check_rows)
display(categorical_check_df)

assert categorical_check_df["matches"].all()

print("PASS: All non-numeric fill values match the training-set mode.")

,column,custom_fill_value,training_mode,matches
0,city,Mumbai,Mumbai,True
1,course,NEET,NEET,True
2,batch_type,Weekday,Weekday,True
3,enrollment_date,2024-10-23,2024-10-23,True
4,income_bracket,5-10L,5-10L,True
5,feedback_text,Need more practice sheets for weak topics,Need more practice sheets for weak topics,True


PASS: All non-numeric fill values match the training-set mode.


# 11. Reflection Questions

### 1. Why must `fit()` be called only on the training split, and never on the full dataset or the test split?

`fit()` learns the values used to replace missing observations, so fitting on the full dataset would allow information from the test set to influence those statistics. That is a form of data leakage because preprocessing would have access to data that is supposed to represent unseen observations. Fitting only on the training split keeps the test set unseen during preprocessing. The fitted imputer can then transform both training and test data consistently using the same learned values.

### 2. `mock_test_3` is MNAR. Does mean/median imputation genuinely solve the problem for this column? What does the `add_missing_indicator` feature contribute that plain imputation does not?

No, mean or median imputation does not genuinely solve the underlying MNAR problem because the probability of missingness is related to the unobserved value itself. Replacing missing `mock_test_3` values with a typical score can distort the distribution and hide the fact that these observations may have systematically low values. The missingness indicator preserves the information that the value was originally missing. A downstream model can therefore learn that missingness itself may carry predictive information, although the indicator does not remove the underlying MNAR bias.

### 3. Suppose a brand-new column, entirely missing in the training data but present in the test data, is passed to your imputer. What does your current implementation do — and what SHOULD a production-grade version do instead?

In the current implementation, a truly new column that was not present during `fit()` causes `transform()` to raise a clear `ValueError`, because the input schema does not match the fitted schema. If a column existed during training but was entirely missing, its mean/median/mode cannot be calculated and the stored fill value becomes `NaN`, so the missing values remain and the issue is explicit rather than silently inventing a statistic. A production-grade implementation should define a clear schema policy and detect all-missing features during fitting, then either reject the feature, drop it, or use a documented domain-specific fallback value.

# 12. Optional Bonus — Per-Column Strategy Overrides (+10%)

The bonus requirement extends the class so individual columns can use a strategy different from the dataset-wide default.

Here:

- `weekly_study_hours` uses **mean** instead of the default median.
- `prev_exam_score` uses **median** explicitly.
- Other columns continue to use their normal default strategies.

This demonstrates different strategies on at least two columns.

In [42]:
# Bonus: use different strategies for individual columns
bonus_imputer = CustomImputer(
    numeric_strategy="median",
    categorical_strategy="most_frequent",
    add_missing_indicator=True,
    column_overrides={
        "weekly_study_hours": "mean",
        "prev_exam_score": "median",
    },
)

bonus_imputer.fit(train_df)
bonus_train = bonus_imputer.transform(train_df)

bonus_demo = pd.DataFrame({
    "column": ["weekly_study_hours", "prev_exam_score"],
    "strategy_used": [
        bonus_imputer.strategies_["weekly_study_hours"],
        bonus_imputer.strategies_["prev_exam_score"],
    ],
    "learned_fill_value": [
        bonus_imputer.fill_values_["weekly_study_hours"],
        bonus_imputer.fill_values_["prev_exam_score"],
    ],
    "expected_value": [
        train_df["weekly_study_hours"].mean(),
        train_df["prev_exam_score"].median(),
    ],
})

display(bonus_demo.round(6))

assert np.isclose(
    bonus_imputer.fill_values_["weekly_study_hours"],
    train_df["weekly_study_hours"].mean()
)

assert np.isclose(
    bonus_imputer.fill_values_["prev_exam_score"],
    train_df["prev_exam_score"].median()
)

assert bonus_train[original_columns].isna().sum().sum() == 0

print("PASS: Per-column strategy overrides work correctly with two different strategies.")

,column,strategy_used,learned_fill_value,expected_value
0,weekly_study_hours,mean,6.0642,6.0642
1,prev_exam_score,median,66.1000,66.1000


PASS: Per-column strategy overrides work correctly with two different strategies.
